# Free-Form KAN — Interpretability Study

Three-phase pipeline to obtain a **sparse, interpretable** B-spline KAN that
learns τ(T, ε) = T:ε without any hard-coded structure.

| Phase | Cell | Technique | Goal |
|---|---|---|---|
| **1** | §8 | Pure L2 (λ=0) | Find any working solution first |
| **2** | §9 | Adaptive sparsity (λ per-step) | Gently push redundant edges to zero |
| **Prune** | §11 | Hard threshold on ‖φ‖ | Commit to the sparse support |
| **3** | §12 | Retrain with masks | Recover accuracy on the sparse subnetwork |

**Architecture:** `[9, 18, 9, 3]` — 2 hidden layers with explicit semantic roles
- **L1 (9→18):** sum/difference of each (T_ij, ε_j) pair  
- **L2 (18→9):** squaring → product via polarization identity  
- **L3 (9→3):** linear assembly ξ_i = Σ_j T_ij ε_j  

**Theoretical sparse support:** 45 / 351 edges = 13% active

**Adaptive λ formula (Phase 2):**  
`lambda_eff = sparse_fraction × task_loss / sparsity_loss`  
This keeps the sparsity term at a fixed fraction of the task loss regardless of scale.

In [ ]:
# §1 — Colab runtime check
import sys, subprocess
import time
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import userdata
    print('Colab secrets available.')

In [ ]:
# §2 — Clone repo and install package
import os
from pathlib import Path

GITHUB_REPO = 'MiRoSi-52wab/Training'
CLONE_DIR   = '/content/ls-kan-fno'

if not Path(CLONE_DIR).exists():
    try:
        token     = userdata.get('GITHUB_TOKEN')
        clone_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
        print('GITHUB_TOKEN found in Colab secrets.')
    except Exception:
        clone_url = f'https://github.com/{GITHUB_REPO}.git'
        print('Warning: GITHUB_TOKEN not set — will fail for private repos.')
    result = subprocess.run(['git', 'clone', clone_url, CLONE_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Clone failed: {result.stderr}')
    print('Cloned successfully.')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print('Pulled latest changes.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', CLONE_DIR, '-q'], check=True)
os.chdir(CLONE_DIR)
print(f'Package installed. CWD = {os.getcwd()}')

In [ ]:
# §3 — Mount Drive and configure paths
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Edit these if your Drive layout differs ───────────────────────────────────
DATA_PATH      = '/content/drive/MyDrive/ls_kan_fno/data/dataset_v3.h5'
RUN_DIR_INTERP = '/content/drive/MyDrive/ls_kan_fno/runs/freeform_kan_interp'
# ─────────────────────────────────────────────────────────────────────────────

Path(RUN_DIR_INTERP).mkdir(parents=True, exist_ok=True)
assert Path(DATA_PATH).exists(), f'Dataset not found at {DATA_PATH}'
print(f'Dataset : {DATA_PATH}')
print(f'Run dir : {RUN_DIR_INTERP}')

In [ ]:
# §4 — GPU check
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — training will still finish in a reasonable time on CPU.')

In [ ]:
# §5 — Build model from interpretability config
from utils.config_loader import load_config, compute_alpha_bounds
from models.freeform_kan import FreeFormKANTauTheta
from datasets.micromechanics import DataLoaderFactory

cfg = load_config('configs/freeformkan_interpretability.yaml')
cfg['train_path'] = DATA_PATH
cfg['val_path']   = DATA_PATH
cfg['output_dir'] = RUN_DIR_INTERP

_am, _ap = compute_alpha_bounds(
    cfg['E_matrix'], cfg.get('nu_matrix', 0.3),
    cfg.get('nu_inclusion', cfg.get('nu_matrix', 0.3)),
    cfg['kappa'], dim=int(cfg.get('dim', 2)),
)
ALPHA0   = (_am + _ap) / 2.0
N_NORMAL = 2

torch.manual_seed(int(cfg.get('seed', 42)))
model = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18, 9]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)

print(model)
print(f'\nLayer param counts : {model.layer_param_counts()}')
print(f'Total params       : {model.n_params():,}')

total_edges = sum(l.n_in * l.n_out for l in model.layers)
print(f'Total edges        : {total_edges}')
print(f'Theoretical active : 45 / {total_edges} = {45/total_edges:.1%}')
print(f'\nα₀ = {ALPHA0:.4g}   eps_input_scale = {model.eps_input_scale}')

In [ ]:
# §6 — Data helpers
from models.freeform_kan import (
    compute_T_mandel, voigt_strain_to_mandel, contraction_loss
)

# Random ε sampling range (from config).
# ε★ from the dataset is CORRELATED with T (both derive from the same microstructure
# via the FFT solver).  Training on (T, ε★) pairs lets the model learn a shortcut
# that fits the correlation manifold instead of the true bilinear law T:ε.
# The bilinearity diagnostic then fails (std/mean ≈ 1, cosine ≈ 0.1).
#
# Fix: sample ε RANDOMLY and independently of T.  Since ξ = T:ε is computed in
# closed form, no FFT data is needed for ε.  The model must learn the true bilinear
# law to minimise the loss on arbitrary (T, ε) pairs.
eps_scale = float(cfg.get('eps_scale', 0.02))   # ε ~ Uniform[-eps_scale, eps_scale]
                                                 # × eps_input_scale=100 → KAN sees [-2, 2]

def prep_batch(batch):
    """
    T_M from real microstructures (realistic T distribution).
    eps_M sampled uniformly at random — independent of T.
    xi_true = T:eps_M computed in closed form (exact label, no FFT needed).
    """
    C_V = batch['C_field'].to(device).double()
    T_M = compute_T_mandel(C_V, ALPHA0, N_NORMAL)      # (B, n, n, H, W)

    B, n, _, H, W = T_M.shape
    eps_M = (torch.rand(B, n, H, W, device=device, dtype=torch.float64) * 2 - 1) * eps_scale

    xi_true = torch.einsum('bijhw,bjhw->bihw', T_M, eps_M)
    return T_M, eps_M, xi_true

def val_eval(m):
    """Evaluate contraction_loss on the full val split (with random ε). Returns scalar."""
    m.eval()
    val_l = []
    with torch.no_grad():
        for vb in loaders['val']:
            T_v, e_v, xi_v = prep_batch(vb)
            val_l.append(float(contraction_loss(m(T_v, e_v), xi_v)))
    m.train()
    return sum(val_l) / max(len(val_l), 1)

loaders  = DataLoaderFactory.from_config(cfg)
sample_b = next(iter(loaders['train']))
T_, e_, xi_ = prep_batch(sample_b)
print(f'T_M    : {tuple(T_.shape)}')
print(f'eps_M  : {tuple(e_.shape)}   (random, independent of T)')
print(f'xi_true: {tuple(xi_.shape)}')
print(f'T range: [{T_.min():.3f}, {T_.max():.3f}]')
print(f'ε range: [{e_.min():.5f}, {e_.max():.5f}]   (uniform in [-{eps_scale}, {eps_scale}])')
print(f'ε×{int(model.eps_input_scale)} KAN input: [{e_.min()*model.eps_input_scale:.2f}, {e_.max()*model.eps_input_scale:.2f}]'
      f'  (x_range = {cfg.get("x_range", [-2.0, 2.0])})')

In [ ]:
# §7 — Smoke test (2 steps)
print('Running 2-step smoke test...')
model.train()
opt_smoke = torch.optim.Adam(model.parameters(), lr=1e-3)
for _ in range(2):
    b = next(iter(loaders['train']))
    T_s, e_s, xi_s = prep_batch(b)
    t0 = time.time()
    opt_smoke.zero_grad()
    xi_pred = model(T_s, e_s)
    loss = contraction_loss(xi_pred, xi_s)
    loss.backward()
    opt_smoke.step()
    dt = time.time() - t0
    print(f'  loss={float(loss):.4e}  {dt*1000:.0f}ms/step')

import math
assert math.isfinite(float(loss)), 'NaN/Inf in loss!'
assert xi_pred.shape == (b['C_field'].shape[0], 3, 64, 64)
print('Smoke test PASSED ✓')

n_steps_total = (int(cfg.get('n_steps_phase1', 10000))
                 + int(cfg.get('n_steps_phase2', 5000))
                 + int(cfg.get('n_steps_phase3', 5000)))
print(f'Total steps (P1+P2+P3): {n_steps_total}')
print(f'Estimated wall-clock  : ~{dt * n_steps_total / 60:.0f} min on this device')

---
## Phase 1 — Pure L2 training

No sparsity penalty.  The model is free to use any combination of edges.
Goal: reach task_loss < 5% so that Phase 2 has a good starting point.

**Why first?**  If we apply sparsity from the start (as in the previous
experiment with fixed λ=0.1), the penalty dominates the task loss early on
and prevents the model from finding a working solution at all.

In [ ]:
# §8 — Phase 1: Pure L2 training (λ=0)
import json
from itertools import cycle

n_steps_p1 = int(cfg.get('n_steps_phase1', 10000))
log_every  = int(cfg.get('log_every', 100))
val_every  = int(cfg.get('val_every', 500))
save_every = int(cfg.get('save_every', 500))

torch.manual_seed(int(cfg.get('seed', 42)))
model = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18, 9]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)

opt_p1 = torch.optim.AdamW(
    model.parameters(), lr=float(cfg.get('lr_phase1', 1e-3)),
    weight_decay=float(cfg.get('weight_decay', 0.0)))
sch_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_p1, T_max=n_steps_p1, eta_min=float(cfg.get('lr_min_phase1', 1e-5)))

CKPT_P1 = Path(RUN_DIR_INTERP) / 'phase1_checkpoint.pt'
HIST_P1 = Path(RUN_DIR_INTERP) / 'phase1_history.jsonl'

start_p1 = 0
hist_p1  = []
if CKPT_P1.exists():
    ck = torch.load(CKPT_P1, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    opt_p1.load_state_dict(ck['optimizer_state_dict'])
    sch_p1.load_state_dict(ck['scheduler_state_dict'])
    start_p1 = int(ck['step']) + 1
    hist_p1  = ck.get('history', [])
    pv = [r['val_loss'] for r in hist_p1 if 'val_loss' in r]
    print(f'Phase 1 resumed from step {start_p1 - 1}'
          f'  (best val: {min(pv):.4e})')
else:
    print(f'Phase 1: starting fresh — {n_steps_p1} steps, λ=0')

train_iter = cycle(loaders['train'])
fh_p1 = open(HIST_P1, 'a')
t0 = time.time()
model.train()

for step in range(start_p1, n_steps_p1):
    T_M, eps_M, xi_true = prep_batch(next(train_iter))
    opt_p1.zero_grad()
    loss_task = contraction_loss(model(T_M, eps_M), xi_true)
    loss_task.backward()
    opt_p1.step(); sch_p1.step()

    row = {'step': step, 'task_loss': float(loss_task),
           'lr': sch_p1.get_last_lr()[0], 'phase': 1}

    if step % val_every == 0:
        row['val_loss'] = val_eval(model)

    hist_p1.append(row)
    fh_p1.write(json.dumps(row) + '\n'); fh_p1.flush()

    if step % log_every == 0:
        done = step - start_p1 + 1
        eta  = (n_steps_p1 - step) * ((time.time() - t0) / max(done, 1))
        vs   = f"  val={row['val_loss']:.3e}" if 'val_loss' in row else ''
        print(f'[P1] {step:5d}/{n_steps_p1}  task={float(loss_task):.3e}{vs}'
              f'  lr={row["lr"]:.1e}  ETA {eta/60:.1f}min')

    if (step + 1) % save_every == 0 or step == n_steps_p1 - 1:
        torch.save({'step': step, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt_p1.state_dict(),
                    'scheduler_state_dict': sch_p1.state_dict(),
                    'history': hist_p1, 'config': cfg}, CKPT_P1)
        print(f'  ✓ Phase 1 checkpoint saved (step {step})')

fh_p1.close()
print(f'\nPhase 1 done — {time.time() - t0:.0f}s')
final_p1 = hist_p1[-1]['task_loss'] if hist_p1 else float('nan')
final_v1 = next((r['val_loss'] for r in reversed(hist_p1) if 'val_loss' in r), float('nan'))
print(f'Final  task={final_p1:.4e}  val={final_v1:.4e}')

---
## Phase 2 — Adaptive sparsity training

**Adaptive λ formula** (fixes the scale-mismatch problem of fixed λ):

$$\lambda_\text{eff} = \alpha \times \frac{\mathcal{L}_\text{task}}{\mathcal{L}_\text{sparse}}$$

where α = `sparse_fraction` (default 0.10).  This guarantees the sparsity term
contributes exactly α × task_loss to the total — regardless of how many active
edges there are or what their magnitudes happen to be.

**Ramp schedule:** α ramps 0 → `sparse_fraction` over the first 25% of Phase 2
steps, giving the model time to adjust without a sudden shock.

In [ ]:
# §9 — Phase 2: Adaptive sparsity training
CKPT_P1 = Path(RUN_DIR_INTERP) / 'phase1_checkpoint.pt'
CKPT_P2 = Path(RUN_DIR_INTERP) / 'phase2_checkpoint.pt'
HIST_P2 = Path(RUN_DIR_INTERP) / 'phase2_history.jsonl'

if not CKPT_P1.exists() and not CKPT_P2.exists():
    raise RuntimeError('Run §8 (Phase 1) before §9.')

n_steps_p2      = int(cfg.get('n_steps_phase2', 5000))
sparse_fraction = float(cfg.get('sparse_fraction', 0.10))
warmup_p2       = n_steps_p2 // 4   # ramp over first 25%

model = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18, 9]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)

opt_p2 = torch.optim.AdamW(
    model.parameters(), lr=float(cfg.get('lr_phase2', 1e-4)),
    weight_decay=float(cfg.get('weight_decay', 0.0)))
sch_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_p2, T_max=n_steps_p2, eta_min=float(cfg.get('lr_min_phase2', 1e-6)))

start_p2 = 0
hist_p2  = []
if CKPT_P2.exists():
    ck = torch.load(CKPT_P2, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    opt_p2.load_state_dict(ck['optimizer_state_dict'])
    sch_p2.load_state_dict(ck['scheduler_state_dict'])
    start_p2 = int(ck['step']) + 1
    hist_p2  = ck.get('history', [])
    print(f'Phase 2 resumed from step {start_p2 - 1}')
else:
    ck_p1 = torch.load(CKPT_P1, map_location=device, weights_only=False)
    model.load_state_dict(ck_p1['model_state_dict'])
    print(f'Phase 2: starting from Phase 1 weights — {n_steps_p2} steps')
    print(f'  sparse_fraction={sparse_fraction}  warmup={warmup_p2} steps')

train_iter = cycle(loaders['train'])
fh_p2 = open(HIST_P2, 'a')
t0 = time.time()
model.train()

for step in range(start_p2, n_steps_p2):
    T_M, eps_M, xi_true = prep_batch(next(train_iter))
    opt_p2.zero_grad()

    xi_pred   = model(T_M, eps_M)
    loss_task = contraction_loss(xi_pred, xi_true)

    # Adaptive lambda with linear ramp
    ramp             = min(1.0, step / max(warmup_p2, 1))
    current_fraction = sparse_fraction * ramp

    x_kan, _, _ = model._flatten_inputs(T_M, eps_M)
    n_sub = min(1024, x_kan.shape[0])
    idx   = torch.randperm(x_kan.shape[0], device=device)[:n_sub]
    loss_sparse = model.sparsity_loss(x_kan[idx].detach())

    # lambda_eff is a plain float — no autograd, just a scaling coefficient
    lambda_eff = (current_fraction * float(loss_task.detach())
                  / (float(loss_sparse.detach()) + 1e-8))
    loss = loss_task + lambda_eff * loss_sparse

    loss.backward()
    opt_p2.step(); sch_p2.step()

    row = {'step': step, 'task_loss': float(loss_task),
           'sparse_weighted': float(lambda_eff * loss_sparse),
           'lambda_eff': float(lambda_eff), 'ramp': ramp,
           'lr': sch_p2.get_last_lr()[0], 'phase': 2}

    if step % val_every == 0:
        row['val_loss'] = val_eval(model)

    hist_p2.append(row)
    fh_p2.write(json.dumps(row) + '\n'); fh_p2.flush()

    if step % log_every == 0:
        done = step - start_p2 + 1
        eta  = (n_steps_p2 - step) * ((time.time() - t0) / max(done, 1))
        vs   = f"  val={row['val_loss']:.3e}" if 'val_loss' in row else ''
        print(f'[P2] {step:5d}/{n_steps_p2}  task={float(loss_task):.3e}'
              f'  sp={float(lambda_eff * loss_sparse):.3e}'
              f'  λ={lambda_eff:.2e}  ramp={ramp:.2f}{vs}'
              f'  lr={row["lr"]:.1e}  ETA {eta/60:.1f}min')

    if (step + 1) % save_every == 0 or step == n_steps_p2 - 1:
        torch.save({'step': step, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt_p2.state_dict(),
                    'scheduler_state_dict': sch_p2.state_dict(),
                    'history': hist_p2, 'config': cfg}, CKPT_P2)
        print(f'  ✓ Phase 2 checkpoint saved (step {step})')

fh_p2.close()
print(f'\nPhase 2 done — {time.time() - t0:.0f}s')
final_p2 = hist_p2[-1]['task_loss'] if hist_p2 else float('nan')
final_v2 = next((r['val_loss'] for r in reversed(hist_p2) if 'val_loss' in r), float('nan'))
print(f'Final  task={final_p2:.4e}  val={final_v2:.4e}')

In [ ]:
# §10 — Training curves: Phase 1 + Phase 2 combined
import matplotlib.pyplot as plt
import numpy as np

CKPT_P1 = Path(RUN_DIR_INTERP) / 'phase1_checkpoint.pt'
CKPT_P2 = Path(RUN_DIR_INTERP) / 'phase2_checkpoint.pt'

# Load histories from checkpoints
def load_history(ckpt_path):
    if not ckpt_path.exists():
        return []
    return torch.load(ckpt_path, map_location='cpu', weights_only=False).get('history', [])

h1 = load_history(CKPT_P1)
h2 = load_history(CKPT_P2)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: task loss (both phases)
ax = axes[0]
if h1:
    s1 = [r['step'] for r in h1]
    ax.semilogy(s1, [r['task_loss'] for r in h1], 'b-', lw=0.8, alpha=0.7, label='P1 task')
    val1 = [(r['step'], r['val_loss']) for r in h1 if 'val_loss' in r]
    if val1:
        vs, vl = zip(*val1)
        ax.semilogy(vs, vl, 'b--', lw=2, marker='o', ms=4, label='P1 val')
if h2:
    s2 = [r['step'] for r in h2]
    # offset Phase 2 x-axis by Phase 1 length for visual continuity
    offset = len(h1)
    ax.semilogy([offset + s for s in s2], [r['task_loss'] for r in h2],
                'r-', lw=0.8, alpha=0.7, label='P2 task')
    val2 = [(r['step'], r['val_loss']) for r in h2 if 'val_loss' in r]
    if val2:
        vs2, vl2 = zip(*val2)
        ax.semilogy([offset + s for s in vs2], vl2,
                    'r--', lw=2, marker='s', ms=4, label='P2 val')
if h1:
    ax.axvline(len(h1), color='gray', ls=':', lw=1, label='P1→P2')
ax.axhline(1e-2, color='orange', ls=':', lw=1.5, label='1%')
ax.axhline(1e-3, color='green',  ls=':', lw=1.5, label='0.1%')
ax.set_xlabel('Step'); ax.set_ylabel('Task rel-L2')
ax.set_title('Task loss — Phases 1 & 2')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Right: sparsity contribution during Phase 2
ax2 = axes[1]
if h2:
    sp_weighted = [r.get('sparse_weighted', 0.0) for r in h2]
    ax2.semilogy(s2, [r['task_loss'] for r in h2], 'r-', lw=0.8, alpha=0.7, label='task')
    ax2.semilogy(s2, [max(v, 1e-10) for v in sp_weighted],
                 'purple', lw=1.2, label='λ·L_sparse')
    ax2.axhline(1e-2, color='orange', ls=':', lw=1.5)
ax2.set_xlabel('Phase 2 step'); ax2.set_ylabel('Loss component')
ax2.set_title('Phase 2 — task vs sparsity contribution')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

plt.tight_layout()
out = Path(RUN_DIR_INTERP) / 'curves_phase1_phase2.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved → {out.name}')
plt.show()

---
## Hard Pruning

For each edge (i→j) in every layer, compute the RMS norm:
$$\|\varphi_{ij}\| = \sqrt{\frac{1}{200} \sum_{x} \varphi_{ij}(x)^2}$$
on a uniform grid of 200 points across `x_range`.

Edges with $\|\varphi_{ij}\| < $ `prune_threshold` are **zeroed out** and
**masked** — they will be held at zero throughout Phase 3 via projected
gradient descent.

Key difference from L1 soft regularization: the decision is **hard and
permanent** within Phase 3.  The optimizer cannot re-activate a pruned edge.

In [ ]:
# §11 — Hard pruning
import numpy as np
import torch.nn.functional as F
from models.freeform_kan import _b_spline_basis

CKPT_P2     = Path(RUN_DIR_INTERP) / 'phase2_checkpoint.pt'
CKPT_PRUNED = Path(RUN_DIR_INTERP) / 'pruned_checkpoint.pt'

if not CKPT_P2.exists():
    raise RuntimeError('Run §9 (Phase 2) before §11.')

prune_threshold = float(cfg.get('prune_threshold', 0.05))
n_k = int(cfg.get('k', 3))
n_G = int(cfg.get('grid_size', 5))

model = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18, 9]),
    grid_size=n_G, k=n_k,
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)
ck_p2 = torch.load(CKPT_P2, map_location=device, weights_only=False)
model.load_state_dict(ck_p2['model_state_dict'])
model.eval()

x_min  = float(model.layers[0].grid[:, n_k].min())
x_max  = float(model.layers[0].grid[:, n_G + n_k].max())
x_plot = torch.linspace(x_min, x_max, 200)

def edge_rms(layer, i, j):
    x_t   = x_plot.unsqueeze(1).to(device).double()
    basis = _b_spline_basis(x_t, layer.grid[i:i+1, :], layer.k)  # (200,1,G+k)
    sp    = (basis[:, 0, :] @ layer.spline_weight[j, i, :]).detach()
    ba    = layer.base_weight[j, i].detach() * F.silu(x_plot.to(device).double())
    return float(((sp + ba) ** 2).mean().sqrt().cpu())

prune_masks  = []
print(f'Pruning threshold: ‖φ‖ < {prune_threshold}')
print()
print(f'{"Layer":<14} {"Total":>7} {"Kept":>7} {"Pruned":>8} {"Active%":>9}')
print('-' * 52)
total_all = kept_all = 0

with torch.no_grad():
    for li, layer in enumerate(model.layers):
        mask = torch.ones(layer.n_out, layer.n_in, dtype=torch.bool)
        for i in range(layer.n_in):
            for j in range(layer.n_out):
                if edge_rms(layer, i, j) < prune_threshold:
                    mask[j, i] = False
                    layer.spline_weight.data[j, i, :] = 0.0
                    layer.base_weight.data[j, i]      = 0.0
        prune_masks.append(mask)
        n_total  = layer.n_in * layer.n_out
        n_kept   = int(mask.sum())
        n_pruned = n_total - n_kept
        total_all += n_total; kept_all += n_kept
        tag = '← output' if li == len(model.layers) - 1 else ''
        print(f'L{li+1}: {layer.n_in}→{layer.n_out:<6}'
              f'  {n_total:>7}  {n_kept:>7}  {n_pruned:>8}'
              f'  {100*n_kept/n_total:>8.1f}%  {tag}')

print('-' * 52)
print(f'{"Total":<14}  {total_all:>7}  {kept_all:>7}  {total_all-kept_all:>8}'
      f'  {100*kept_all/total_all:>8.1f}%')
print()
print(f'Theory target: 45 / 351 = 12.8% active')

# Per-layer norm distributions
print()
print('── Edge norm distributions (before pruning) ──')
for li, layer in enumerate(model.layers):
    norms = np.array([edge_rms(layer, i, j)
                      for i in range(layer.n_in) for j in range(layer.n_out)])
    print(f'  L{li+1}: min={norms.min():.3f}  p25={np.percentile(norms,25):.3f}'
          f'  median={np.median(norms):.3f}  p75={np.percentile(norms,75):.3f}'
          f'  max={norms.max():.3f}')

# Save pruned model and masks
torch.save({'model_state_dict': model.state_dict(),
            'prune_masks':      [m.cpu() for m in prune_masks],
            'prune_threshold':  prune_threshold,
            'config':           cfg}, CKPT_PRUNED)
print(f'\nPruned model + masks saved → {CKPT_PRUNED.name}')

---
## Phase 3 — Retrain with masks (projected gradient descent)

Pruned edges are held at **exactly zero** throughout:
1. After `loss.backward()`: zero the **gradients** of pruned edges
2. After `optimizer.step()`: zero the **weights** of pruned edges

Step 1 prevents the optimizer from accumulating gradient signal for those edges.
Step 2 is needed because AdamW momentum can still slightly update the weight
even with zero gradient — zeroing afterwards projects back onto the constraint.

In [ ]:
# §12 — Phase 3: Retrain with masks (projected gradient descent)
CKPT_PRUNED = Path(RUN_DIR_INTERP) / 'pruned_checkpoint.pt'
CKPT_P3     = Path(RUN_DIR_INTERP) / 'phase3_checkpoint.pt'
HIST_P3     = Path(RUN_DIR_INTERP) / 'phase3_history.jsonl'

if not CKPT_PRUNED.exists():
    raise RuntimeError('Run §11 (pruning) before §12.')

n_steps_p3 = int(cfg.get('n_steps_phase3', 5000))

ck_pr = torch.load(CKPT_PRUNED, map_location=device, weights_only=False)
model = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18, 9]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)
model.load_state_dict(ck_pr['model_state_dict'])
prune_masks = [m.to(device) for m in ck_pr['prune_masks']]

n_kept  = sum(m.sum().item() for m in prune_masks)
n_total = sum(m.numel() for m in prune_masks)
print(f'Pruned model: {n_kept}/{n_total} edges active ({100*n_kept/n_total:.1f}%)')

opt_p3 = torch.optim.AdamW(
    model.parameters(), lr=float(cfg.get('lr_phase3', 1e-4)),
    weight_decay=float(cfg.get('weight_decay', 0.0)))
sch_p3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_p3, T_max=n_steps_p3, eta_min=float(cfg.get('lr_min_phase3', 1e-6)))

start_p3 = 0
hist_p3  = []
if CKPT_P3.exists():
    ck = torch.load(CKPT_P3, map_location=device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    opt_p3.load_state_dict(ck['optimizer_state_dict'])
    sch_p3.load_state_dict(ck['scheduler_state_dict'])
    prune_masks = [m.to(device) for m in ck['prune_masks']]
    start_p3 = int(ck['step']) + 1
    hist_p3  = ck.get('history', [])
    print(f'Phase 3 resumed from step {start_p3 - 1}')
else:
    print(f'Phase 3: retraining sparse model — {n_steps_p3} steps, λ=0')

train_iter = cycle(loaders['train'])
fh_p3 = open(HIST_P3, 'a')
t0 = time.time()
model.train()

for step in range(start_p3, n_steps_p3):
    T_M, eps_M, xi_true = prep_batch(next(train_iter))
    opt_p3.zero_grad()

    loss_task = contraction_loss(model(T_M, eps_M), xi_true)
    loss_task.backward()

    # ── 1. Zero gradients of pruned edges ─────────────────────────────────
    with torch.no_grad():
        for layer, mask in zip(model.layers, prune_masks):
            if layer.spline_weight.grad is not None:
                layer.spline_weight.grad[~mask] = 0.0
            if layer.base_weight.grad is not None:
                layer.base_weight.grad[~mask] = 0.0

    opt_p3.step(); sch_p3.step()

    # ── 2. Project: zero pruned weights (counters AdamW momentum) ─────────
    with torch.no_grad():
        for layer, mask in zip(model.layers, prune_masks):
            layer.spline_weight.data[~mask] = 0.0
            layer.base_weight.data[~mask]   = 0.0

    row = {'step': step, 'task_loss': float(loss_task),
           'lr': sch_p3.get_last_lr()[0], 'phase': 3}

    if step % val_every == 0:
        row['val_loss'] = val_eval(model)

    hist_p3.append(row)
    fh_p3.write(json.dumps(row) + '\n'); fh_p3.flush()

    if step % log_every == 0:
        done = step - start_p3 + 1
        eta  = (n_steps_p3 - step) * ((time.time() - t0) / max(done, 1))
        vs   = f"  val={row['val_loss']:.3e}" if 'val_loss' in row else ''
        print(f'[P3] {step:5d}/{n_steps_p3}  task={float(loss_task):.3e}{vs}'
              f'  lr={row["lr"]:.1e}  ETA {eta/60:.1f}min')

    if (step + 1) % save_every == 0 or step == n_steps_p3 - 1:
        torch.save({'step': step, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': opt_p3.state_dict(),
                    'scheduler_state_dict': sch_p3.state_dict(),
                    'prune_masks': [m.cpu() for m in prune_masks],
                    'history': hist_p3, 'config': cfg}, CKPT_P3)
        print(f'  ✓ Phase 3 checkpoint saved (step {step})')

fh_p3.close()
print(f'\nPhase 3 done — {time.time() - t0:.0f}s')
final_p3 = hist_p3[-1]['task_loss'] if hist_p3 else float('nan')
final_v3 = next((r['val_loss'] for r in reversed(hist_p3) if 'val_loss' in r), float('nan'))
print(f'Final  task={final_p3:.4e}  val={final_v3:.4e}')

In [ ]:
# §13 — Training curves: all three phases
import matplotlib.pyplot as plt
import numpy as np

def load_history(path):
    p = Path(path)
    if not p.exists():
        return []
    return torch.load(p, map_location='cpu', weights_only=False).get('history', [])

h1 = load_history(Path(RUN_DIR_INTERP) / 'phase1_checkpoint.pt')
h2 = load_history(Path(RUN_DIR_INTERP) / 'phase2_checkpoint.pt')
h3 = load_history(Path(RUN_DIR_INTERP) / 'phase3_checkpoint.pt')

fig, ax = plt.subplots(figsize=(11, 4))

offset2 = len(h1)
offset3 = offset2 + len(h2)

for h, off, col, label in [
    (h1, 0,       'steelblue', 'Phase 1 (pure L2)'),
    (h2, offset2, 'crimson',   'Phase 2 (adaptive λ)'),
    (h3, offset3, 'seagreen',  'Phase 3 (masked retrain)'),
]:
    if not h:
        continue
    steps = [off + r['step'] for r in h]
    ax.semilogy(steps, [r['task_loss'] for r in h],
                color=col, lw=0.7, alpha=0.6, label=f'{label} train')
    val_pts = [(off + r['step'], r['val_loss']) for r in h if 'val_loss' in r]
    if val_pts:
        vs, vl = zip(*val_pts)
        ax.semilogy(vs, vl, color=col, lw=2, ls='--', marker='o', ms=4)

for off, label in [(offset2, 'P1→P2'), (offset3, 'P2→P3 (prune)')]:
    if off > 0:
        ax.axvline(off, color='gray', ls=':', lw=1.2, label=label)

ax.axhline(1e-2, color='orange', ls=':', lw=1.5, label='1%')
ax.axhline(1e-3, color='green',  ls=':', lw=1.5, label='0.1%')
ax.set_xlabel('Cumulative step')
ax.set_ylabel('Task rel-L2 error')
ax.set_title('Free-form KAN interpretability — three-phase training')
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = Path(RUN_DIR_INTERP) / 'curves_all_phases.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved → {out.name}')
plt.show()

In [ ]:
# §14 — Edge function visualization (loads Phase 3 checkpoint)
#
# Plots every φᵢⱼ(x) per layer with the RMS norm ‖φ‖ annotated.
# Pruned edges (‖φ‖ ≈ 0) should appear as flat zero lines.
# Active edges should resemble linear (L1), quadratic (L2), or linear (L3) functions.
import torch.nn.functional as F
from models.freeform_kan import _b_spline_basis

CKPT_P3 = Path(RUN_DIR_INTERP) / 'phase3_checkpoint.pt'
CKPT_P2 = Path(RUN_DIR_INTERP) / 'phase2_checkpoint.pt'  # fallback
_vis_ckpt = CKPT_P3 if CKPT_P3.exists() else CKPT_P2
if not _vis_ckpt.exists():
    raise FileNotFoundError('No checkpoint found. Run §9 or §12 first.')

_c = torch.load(_vis_ckpt, map_location=device, weights_only=False)
vis_m = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18, 9]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)
vis_m.load_state_dict(_c['model_state_dict'])
vis_m.eval()
print(f'Loaded: {vis_m}  (from {_vis_ckpt.name})')

n_k = int(cfg.get('k', 3))
n_G = int(cfg.get('grid_size', 5))
x_min  = float(vis_m.layers[0].grid[:, n_k].min())
x_max  = float(vis_m.layers[0].grid[:, n_G + n_k].max())
x_plot = torch.linspace(x_min, x_max, 200)

def eval_edge(layer, i, j):
    x_t   = x_plot.unsqueeze(1).to(device).double()
    basis = _b_spline_basis(x_t, layer.grid[i:i+1, :], layer.k)
    sp    = (basis[:, 0, :] @ layer.spline_weight[j, i, :]).detach()
    ba    = layer.base_weight[j, i].detach() * F.silu(x_plot.to(device).double())
    return (sp + ba).cpu().float().numpy()

_n  = 3
_s  = float(cfg.get('eps_input_scale', 100.0))
_T_labels   = [f'T{i}{j}' for i in range(_n) for j in range(i, _n)]
_eps_labels = [f'e{k}x{_s:.0f}' for k in range(_n)]
_input_labels = _T_labels + _eps_labels   # 9 total

for li, layer in enumerate(vis_m.layers):
    n_in, n_out = layer.n_in, layer.n_out
    row_labels  = _input_labels if li == 0 else [f'h{i}' for i in range(n_in)]
    out_labels  = ([f'h{j}' for j in range(n_out)] if li < len(vis_m.layers) - 1
                   else ['xi0', 'xi1', 'xi2'])

    fig, axes = plt.subplots(
        n_in, n_out,
        figsize=(max(n_out * 0.9, 8), max(n_in * 0.9, 8)),
        sharex=True,
    )
    if n_in == 1: axes = axes[np.newaxis, :]
    if n_out == 1: axes = axes[:, np.newaxis]

    phase_tag = 'after pruning' if li == 0 and CKPT_P3.exists() else ''
    fig.suptitle(
        f'L{li+1}: {n_in}\u2192{n_out}  ({n_in*n_out} edges)  {phase_tag}\n'
        f'Row=input  Col=output  \u2016\u03c6\u2016 annotated per cell',
        fontsize=9,
    )
    for i in range(n_in):
        for j in range(n_out):
            ax = axes[i, j]
            y  = eval_edge(layer, i, j)
            rms = float(np.sqrt(np.mean(y ** 2)))
            col = 'tomato' if rms < 0.05 else 'steelblue'
            ax.plot(x_plot.numpy(), y, lw=0.7, color=col)
            ax.axhline(0, color='k', lw=0.3, ls='--')
            ax.set_xticks([]); ax.set_yticks([])
            if j == 0:
                ax.set_ylabel(row_labels[i], fontsize=5, labelpad=1)
            if i == 0:
                ax.set_title(out_labels[j], fontsize=5, pad=1)
            ax.text(0.97, 0.95, f'{rms:.2f}', transform=ax.transAxes,
                    fontsize=4, ha='right', va='top', color='gray')
    plt.tight_layout()
    out_l = Path(RUN_DIR_INTERP) / f'edges_L{li+1}.png'
    plt.savefig(out_l, dpi=150, bbox_inches='tight')
    print(f'L{li+1} edge grid saved → {out_l.name}')
    plt.show(); plt.close(fig)

# Sparsity summary
print()
print('── Active edge summary (‖φ‖ ≥ 0.05) ──')
total_e = total_a = 0
for li, layer in enumerate(vis_m.layers):
    norms = np.array([float(np.sqrt(np.mean(eval_edge(layer, i, j)**2)))
                      for i in range(layer.n_in) for j in range(layer.n_out)])
    n_act = (norms >= 0.05).sum()
    total_e += len(norms); total_a += n_act
    print(f'  L{li+1}: {n_act}/{len(norms)} active ({100*n_act/len(norms):.0f}%)')
print(f'  All: {total_a}/{total_e} active ({100*total_a/total_e:.0f}%)')
print(f'  Theory: 45/351 = 12.8%')

In [ ]:
# §15 — Bilinearity diagnostic: mixed Hessian ∂²ξ / ∂T ∂ε
#
# For exact T:ε, the mixed second-order derivative is constant:
#   ∂²ξ_i / ∂T_{jk} ∂ε_l = δ_ij δ_kl  (scaled by eps_input_scale)
# Two checks: (1) constancy across voxels, (2) cosine sim to δ_ij δ_kl.

def mixed_hessian_at_voxel(m, T_vox, eps_vox):
    n = eps_vox.shape[0]
    T_in = T_vox.reshape(1, n, n, 1, 1).clone().requires_grad_(True)
    e_in = eps_vox.reshape(1, n, 1, 1).clone().requires_grad_(True)
    xi = m(T_in, e_in)
    H  = torch.zeros(n, n, n, n)
    for i in range(n):
        g_T = torch.autograd.grad(xi[0,i,0,0], T_in, create_graph=True, retain_graph=True)[0]
        for j in range(n):
            for kk in range(n):
                H_row = torch.autograd.grad(g_T[0,j,kk,0,0], e_in, retain_graph=True)[0]
                H[i, j, kk, :] = H_row[0, :, 0, 0].detach()
    return H

n_c       = 3
eps_scale = float(model.eps_input_scale)
H_expected = torch.zeros(n_c, n_c, n_c, n_c)
for i in range(n_c):
    for kk in range(n_c):
        H_expected[i, i, kk, kk] = eps_scale

N_VOXELS = 6
torch.manual_seed(42)
val_batch = next(iter(loaders['val']))
T_val, eps_val, _ = prep_batch(val_batch)
B, _, H_, W_ = eps_val.shape

model.eval()
H_samples = []
for _ in range(N_VOXELS):
    b  = torch.randint(0, B, (1,)).item()
    hh = torch.randint(0, H_, (1,)).item()
    ww = torch.randint(0, W_, (1,)).item()
    H_samples.append(mixed_hessian_at_voxel(model, T_val[b,:,:,hh,ww], eps_val[b,:,hh,ww]))

H_stack = torch.stack(H_samples)
H_mean  = H_stack.mean(0)
H_std   = H_stack.std(0)
constancy = H_std.norm() / H_mean.norm().clamp(min=1e-12)
cos_sim   = ((H_mean.flatten() @ H_expected.flatten())
             / (H_mean.norm() * H_expected.norm()).clamp(min=1e-12))

print('=' * 60)
print('Bilinearity Diagnostic')
print('=' * 60)
print(f'Expected: H[i,j,k,l] = {eps_scale:.0f} x delta_ij x delta_kl')
print()
threshold = eps_scale * 0.05
for i in range(n_c):
    for j in range(n_c):
        for kk in range(n_c):
            for ll in range(n_c):
                v  = H_mean[i,j,kk,ll].item()
                ex = H_expected[i,j,kk,ll].item()
                if abs(v) > threshold:
                    print(f'  H[{i},{j},{kk},{ll}] = {v:+7.2f}  expected {ex:+7.1f}')
print()
print(f'Constancy (std/mean): {constancy:.4f}')
print('  < 0.10 -> bilinear ✓' if constancy < 0.10 else '  > 0.10 -> higher-order ✗')
print(f'Pattern cosine sim : {cos_sim:.4f}')
print('  > 0.99 -> correct structure ✓' if cos_sim > 0.99 else '  < 0.99 -> differs from T:eps ✗')
bilinear_ok = constancy < 0.10 and cos_sim > 0.99
print(f'\nOverall: {"PASS ✓" if bilinear_ok else "FAIL ✗"}')
print('=' * 60)

In [ ]:
# §16 — Embed in LSFNO and compare to FFT ground truth
from models.ls_fno import LSFNO
from utils.config_loader import load_config as _lc

model.eval()
_lsfno_cfg = _lc('configs/freeformkan_interpretability.yaml')
_lsfno_cfg['train_path'] = DATA_PATH
lsfno_int = LSFNO.from_config(_lsfno_cfg, tau_theta=model).to(device)
lsfno_int.eval()

test_loader = torch.utils.data.DataLoader(
    DataLoaderFactory.from_config(cfg)['test'].dataset,
    batch_size=8, shuffle=False, num_workers=0,
)

errors = []
with torch.no_grad():
    for tb in test_loader:
        C  = tb['C_field'].to(device)
        eb = tb['eps_bar'].to(device)
        ep = tb['eps_star'].to(device)
        ep_pred = lsfno_int(C, eb, use_checkpointing=False)
        diff  = (ep_pred - ep).reshape(ep.shape[0], -1)
        ref   = ep.reshape(ep.shape[0], -1)
        errors.extend((diff.norm(dim=1) / ref.norm(dim=1).clamp(min=1e-12)).cpu().tolist())

import numpy as np
errors = np.array(errors)
print('=' * 55)
print('Embedded LSFNO (interpretable KAN) vs FFT')
print('=' * 55)
print(f'Samples : {len(errors)}')
print(f'Mean    : {errors.mean():.4%}')
print(f'Median  : {np.median(errors):.4%}')
print(f'p90     : {np.percentile(errors, 90):.4%}')
print(f'Max     : {errors.max():.4%}')
passed = errors.mean() < 1e-3
print(f'Passed (mean < 0.1%): {"✓" if passed else "✗"}')
print('=' * 55)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(errors * 100, bins=30, color='steelblue', edgecolor='white', lw=0.4)
ax.axvline(errors.mean() * 100, color='red', ls='--', lw=1.5,
           label=f'Mean {errors.mean():.3%}')
ax.axvline(0.1, color='orange', ls=':', lw=1.5, label='0.1% threshold')
ax.set_xlabel('Relative L2 field error (%)'); ax.set_ylabel('Count')
ax.set_title('Embedded evaluation: interpretable KAN vs FFT')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = Path(RUN_DIR_INTERP) / 'embedded_errors.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved → {out.name}')
plt.show()